In [0]:
#criando os chemas dentro do workspace
#spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.drs_bronze")
#spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.drs_silver")
#spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.drs_gold")

In [0]:
# Definindo o caminho do nosso arquivo e criando o dataframe
# file_path = "/Volumes/workspace/pipeline_estudo/raw_files/"

# df = (
#     spark.read
#     .option("header", True)
#     .option("inferSchema", False)
#     .option("multiLine", True)
#     .option("quote", '"')
#     .option("escape", '"')
#     .csv(file_path)
# )

# display(df)

# df.write.format("delta") \
#     .mode("overwrite") \
#     .saveAsTable("workspace.drs_bronze.sales")

file_path = "/Volumes/workspace/pipeline_estudo/raw_files/sales.csv"

df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("multiLine", True)
    .option("quote", '"')
    .option("escape", '"')
    .csv(file_path)
)

display(df)

df = df.toDF(*[c.replace(" ", "_").replace("á","a").replace("é","e").replace("í","i").replace("ó","o").replace("ú","u").replace("ã","a").replace("ç","c") for c in df.columns])

df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("drs_bronze_sales")

In [0]:
# importar bibliotecas
import re
import unicodedata

In [0]:
# Criar função de limpeza
def clean_column_name(col_name):

    # Remover acentos
    col_name = unicodedata.normalize('NFKD', col_name)
    col_name = col_name.encode('ascii', 'ignore').decode('utf-8')

    # Minúsculo
    col_name = col_name.lower()

    # Substituir espaços por _
    col_name = col_name.replace(" ", "_")

    # Remover caracteres especiais
    col_name = re.sub(r'[^a-z0-9_]', '', col_name)

    return col_name

In [0]:
# Aplicar no dataframe
df = df.toDF(*[clean_column_name(c) for c in df.columns])
display(df)


In [0]:
print(df.columns)

In [0]:
# Salvar tabela bronze sales
df.write.format("delta") \
.mode("overwrite") \
.saveAsTable("workspace.drs_bronze.sales")


In [0]:
# Verificando a estrututra da tabela
df = spark.sql("""
          select * from workspace.drs_bronze.sales
          """)
display(df)

In [0]:
df.printSchema()

## De agora em diante vamos transformar nossa pipeline em PRODUCTION-GRADE PIPELINE

Com:

✔ Ingestão incremental
✔ Delta MERGE
✔ idempotência
✔ schema evolution
✔ data quality checks

### Estratégia

Vamos criar uma v2 com estes pilares:

- ambiente paralelo, sem sobrescrever o que já funciona
- carga incremental, em vez de overwrite cego
- MERGE/UPSERT na Silver
- controles de auditoria, como data de ingestão
- qualidade de dados
- job separado para testes

### Como ficará a nova arquitetura

Vamos manter a atual:

- workspace.drs_bronze.sales
- workspace.drs_silver.sales
- workspace.drs_gold.*

E criar uma nova trilha:

- workspace.drs_bronze.sales_v2
- workspace.drs_silver.sales_v2
- workspace.drs_gold.sales_by_region_v2
- workspace.drs_gold.sales_by_category_v2
- workspace.drs_gold.top_products_v2

### O que muda da pipeline atual para a production-grade

Hoje você faz, em essência:

read csv
- overwrite bronze
- overwrite silver
- overwrite gold

Na v2 vamos caminhar para:

read csv
- append/ingest bronze com metadados
- merge silver por chave de negócio
- rebuild ou merge gold

In [0]:
# Primeiro, vamos adicionar metadados de ingestão.
from pyspark.sql.functions import current_timestamp, lit

In [0]:
# Criando um novo dataframe a partir do df já padronizado

df_bronze_v2 = (
    df
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_file", lit("sales.csv"))
)

In [0]:
# Salvando a tabela sales drs_bronze.sales_v2

df_bronze_v2.write \
.format("delta") \
.mode("append") \
.saveAsTable("workspace.drs_bronze.sales_v2")


In [0]:
# Validando a tabela drs_bronze.sales_v2
spark.sql("""
SELECT *
FROM workspace.drs_bronze.sales_v2
LIMIT 10
""").display()

In [0]:
# Contando registros da tabela sales_v2 na bronze
spark.sql("""
SELECT COUNT(*) AS total_bronze_v2
FROM workspace.drs_bronze.sales_v2
""").display()


In [0]:
# Verificar lotes por arquivo e ingestão
spark.sql("""
SELECT
    source_file,
    date_trunc('minute', ingestion_timestamp) AS ingestion_minute,
    COUNT(*) AS total_rows
FROM workspace.drs_bronze.sales_v2
GROUP BY source_file, date_trunc('minute', ingestion_timestamp)
ORDER BY ingestion_minute DESC
""").display()

In [0]:
# Validar duplicidade por chave de negócio
spark.sql("""
SELECT
    id_do_pedido,
    id_do_produto,
    COUNT(*) AS qtd
FROM workspace.drs_bronze.sales_v2
GROUP BY id_do_pedido, id_do_produto
HAVING COUNT(*) > 1
ORDER BY qtd DESC
""").display()